# Fine-tune OCR hóa đơn (VietOCR + PaddleOCR) — TownHub

Notebook chạy **hết một mạch** trên Google Colab (bật GPU: *Runtime → Change runtime type → T4 GPU*):
1. Sinh dataset hóa đơn tiếng Việt synthetic (không cần gán tay).
2. Fine-tune **VietOCR** (recognition).
3. Fine-tune **PaddleOCR** (detection + recognition).
4. Export weights + hướng dẫn nạp vào service 3 engine.

> Engine `gemini` dùng API, KHÔNG cần train. Notebook này lo 2 engine tự train.


## 0. Kiểm tra GPU + lấy mã nguồn


In [ ]:
!nvidia-smi -L || echo '⚠️ Chưa bật GPU — vào Runtime → Change runtime type → T4 GPU'

REPO_URL = 'https://github.com/thuongerikdev/TownHub'  # đổi nếu repo của bạn khác
import os
if not os.path.exists('townhub'):
    !git clone --depth 1 $REPO_URL townhub
%cd townhub/ocr-service
!ls training


## 1. Sinh dataset synthetic
Render hóa đơn GTGT giả + tự ghi nhãn detection & recognition. `--n` = số hóa đơn.
Thêm `--photo --marks` nếu muốn đa dạng (ảnh chụp trên bàn, watermark…).


In [ ]:
!apt-get -qq install -y fonts-dejavu-core >/dev/null
!pip -q install pillow numpy opencv-python-headless
!python training/make_dataset.py --n 800 --out ./dataset --fonts /usr/share/fonts/truetype/dejavu


In [ ]:
# Xem thử 1 hóa đơn + vài crop
from PIL import Image
import glob
display(Image.open('dataset/det/images/inv_00001.jpg'))
for f in sorted(glob.glob('dataset/rec/images/inv_00001_*.jpg'))[:5]:
    display(Image.open(f))


## 2. Fine-tune VietOCR (recognition)
Fine-tune từ pretrain `vgg_transformer` (đúng backbone service đang dùng).


In [ ]:
!pip -q install vietocr==0.3.13


In [ ]:
!python training/finetune_vietocr.py --data ./dataset/rec --iters 15000 \
        --batch 32 --out ./weights/vietocr_invoice.pth
print('✅ VietOCR weights: ocr-service/weights/vietocr_invoice.pth')


## 3. Fine-tune PaddleOCR (detection + recognition)


In [ ]:
!pip -q install paddlepaddle-gpu==2.6.1
!pip -q install paddleocr==2.7.3
import os
if not os.path.exists('PaddleOCR'):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git


### 3.1. Tải model pretrain (để fine-tune, không train from scratch)


In [ ]:
%cd PaddleOCR
!mkdir -p pretrain && cd pretrain \
 && wget -q https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_rec_train.tar \
 && wget -q https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_det_train.tar \
 && tar xf ch_PP-OCRv4_rec_train.tar && tar xf ch_PP-OCRv4_det_train.tar
!ls pretrain


### 3.2. Train RECOGNITION
Dùng `dataset/rec` + `dataset/dict_vi.txt`. Đường dẫn dữ liệu là tương đối từ thư mục PaddleOCR (`../dataset/...`).


In [ ]:
!python tools/train.py -c configs/rec/PP-OCRv4/PP-OCRv4_rec.yml \
  -o Global.pretrained_model=./pretrain/ch_PP-OCRv4_rec_train/best_accuracy \
     Global.character_dict_path=../dataset/dict_vi.txt \
     Global.use_space_char=True \
     Global.epoch_num=80 \
     Global.save_model_dir=./output/rec_vi \
     Train.dataset.data_dir=../dataset/rec \
     Train.dataset.label_file_list=['../dataset/rec/train.txt'] \
     Eval.dataset.data_dir=../dataset/rec \
     Eval.dataset.label_file_list=['../dataset/rec/val.txt']


### 3.3. Train DETECTION


In [ ]:
!python tools/train.py -c configs/det/PP-OCRv4/PP-OCRv4_det_student.yml \
  -o Global.pretrained_model=./pretrain/ch_PP-OCRv4_det_train/best_accuracy \
     Global.epoch_num=150 \
     Global.save_model_dir=./output/det_vi \
     Train.dataset.data_dir=../dataset/det \
     Train.dataset.label_file_list=['../dataset/det/train_label.txt'] \
     Eval.dataset.data_dir=../dataset/det \
     Eval.dataset.label_file_list=['../dataset/det/val_label.txt']


### 3.4. Export sang inference model (BẮT BUỘC để service dùng)


In [ ]:
!python tools/export_model.py -c configs/rec/PP-OCRv4/PP-OCRv4_rec.yml \
  -o Global.pretrained_model=./output/rec_vi/best_accuracy \
     Global.save_inference_dir=./inference/rec_vi
!python tools/export_model.py -c configs/det/PP-OCRv4/PP-OCRv4_det_student.yml \
  -o Global.pretrained_model=./output/det_vi/best_accuracy \
     Global.save_inference_dir=./inference/det_vi
%cd ..
print('✅ Paddle inference: PaddleOCR/inference/rec_vi & det_vi')


## 4. Lưu weights về Google Drive (khỏi mất khi Colab tắt)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/townhub_ocr
!cp -r weights /content/drive/MyDrive/townhub_ocr/ 2>/dev/null; true
!cp -r PaddleOCR/inference /content/drive/MyDrive/townhub_ocr/ 2>/dev/null; true
!cp dataset/dict_vi.txt /content/drive/MyDrive/townhub_ocr/ 2>/dev/null; true
print('✅ Đã lưu vào Drive/townhub_ocr')


## 5. Chạy service 3 engine với weights vừa train (tùy chọn)
Trỏ biến môi trường sang weights fine-tune rồi mở tunnel cloudflared để backend .NET gọi vào.


In [ ]:
!pip -q install fastapi uvicorn pydantic requests pdf2image google-generativeai torch easyocr vietocr
!pip -q install paddlepaddle-gpu==2.6.1 paddleocr==2.7.3
import os
os.environ['VIETOCR_WEIGHTS'] = '/content/townhub/ocr-service/weights/vietocr_invoice.pth'
os.environ['PADDLE_REC_DIR']  = '/content/townhub/ocr-service/PaddleOCR/inference/rec_vi'
os.environ['PADDLE_DET_DIR']  = '/content/townhub/ocr-service/PaddleOCR/inference/det_vi'
os.environ['PADDLE_REC_DICT'] = '/content/townhub/ocr-service/dataset/dict_vi.txt'
os.environ['GEMINIKEY'] = 'DAN_KEY_GEMINI_CUA_BAN'  # nếu dùng engine gemini
os.environ['OCRKEY']    = 'doan-ocr-2026'          # khớp OCR_API_KEY phía .NET
print('Đã set env. Chạy 2 cell dưới để mở tunnel + service.')


In [ ]:
# Tunnel cloudflared -> lấy URL https công khai đặt vào OCR_SERVICE_URL của backend
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess, time, re
proc = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:7860','--no-autoupdate'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url=None
for line in proc.stdout:
    print(line, end='')
    m=re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m: url=m.group(0); break
print('\n🌐 OCR_SERVICE_URL =', url)


In [ ]:
# Chạy service (giữ cell này chạy). Backend .NET đặt OCR_SERVICE_URL = URL ở trên.
!python app.py
